<a href="https://colab.research.google.com/github/uwol1116/GAI-Class/blob/main/02-Practice%20Problems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Practice Problem 1

import torch
import torch.nn as nn
from datasets import load_dataset
from collections import Counter

dataset = load_dataset("imdb")

def simple_tokenize(text):
    return text.lower().split()

counter = Counter()
for text in dataset['train']['text'][:5000]:
    counter.update(simple_tokenize(text))

vocab = {word: i + 2 for i, (word, _) in enumerate(counter.most_common(10000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def text_to_indices(text):
    return [vocab.get(word, 1) for word in simple_tokenize(text)]


class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (h_n, c_n) = self.lstm(embedded)

        final_hidden_state = h_n[-1]

        print(f"[Model] Hidden state shape: {final_hidden_state.shape}")

        logit = self.fc(final_hidden_state)
        return logit

def manual_sigmoid(x):
    return 1.0 / (1.0 + torch.exp(-x))

vocab_size = len(vocab)
embed_dim = 32
hidden_dim = 64

model = SentimentLSTM(vocab_size, embed_dim, hidden_dim)
model.eval()


SentimentLSTM(
  (embedding): Embedding(10002, 32)
  (lstm): LSTM(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [6]:
test_samples = dataset['test']

for i in range(2):
    text = test_samples['text'][i]
    true_label = test_samples['label'][i]

    indices = text_to_indices(text)

    input_tensor = torch.tensor([indices], dtype=torch.long)

    with torch.no_grad():
        print(f"\n--- Test Sample {i+1} ---")
        logit = model(input_tensor)
        prob = manual_sigmoid(logit).item()
        pred_label = 1 if prob >= 0.5 else 0

    print(f"Prediction: {prob:.4f} -> Class {pred_label}")
    print(f"True Label: {true_label}")


--- Test Sample 1 ---
[Model] Hidden state shape: torch.Size([1, 64])
Prediction: 0.5461 -> Class 1
True Label: 0

--- Test Sample 2 ---
[Model] Hidden state shape: torch.Size([1, 64])
Prediction: 0.5131 -> Class 1
True Label: 0


In [7]:
#Practice Problem 2

import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=2, padding=1)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1)
        self.relu2 = nn.ReLU()

        self.fc = nn.Linear(in_features=32 * 7 * 7, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        print(f"[Layer 1] Shape after Conv1+ReLU: {x.shape}")

        x = self.conv2(x)
        x = self.relu2(x)
        print(f"[Layer 2] Shape after Conv2+ReLU: {x.shape}")

        x = x.view(x.size(0), -1)

        x = self.fc(x)
        print(f"[Layer 3] Shape after Linear layer: {x.shape}")

        return x

model = SimpleCNN()
model.eval()

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (relu1): ReLU()
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (relu2): ReLU()
  (fc): Linear(in_features=1568, out_features=10, bias=True)
)

In [8]:
for i in range(3):
    image, true_label = test_dataset[i]
    image_tensor = image.unsqueeze(0)

    print(f"\n--- Test Image {i+1} ---")
    with torch.no_grad():
        logits = model(image_tensor)

        predicted_label = torch.argmax(logits, dim=1).item()

    print(f">> Prediction: {predicted_label}")
    print(f">> True Label: {true_label}")


--- Test Image 1 ---
[Layer 1] Shape after Conv1+ReLU: torch.Size([1, 16, 14, 14])
[Layer 2] Shape after Conv2+ReLU: torch.Size([1, 32, 7, 7])
[Layer 3] Shape after Linear layer: torch.Size([1, 10])
>> Prediction: 5
>> True Label: 7

--- Test Image 2 ---
[Layer 1] Shape after Conv1+ReLU: torch.Size([1, 16, 14, 14])
[Layer 2] Shape after Conv2+ReLU: torch.Size([1, 32, 7, 7])
[Layer 3] Shape after Linear layer: torch.Size([1, 10])
>> Prediction: 5
>> True Label: 2

--- Test Image 3 ---
[Layer 1] Shape after Conv1+ReLU: torch.Size([1, 16, 14, 14])
[Layer 2] Shape after Conv2+ReLU: torch.Size([1, 32, 7, 7])
[Layer 3] Shape after Linear layer: torch.Size([1, 10])
>> Prediction: 5
>> True Label: 1
